In [ ]:
import requests
import json

def opening(file_path, url):
    with open(file_path, "rb") as f:
        file_data = f.read()
    files = {"file": (file_path, file_data, "application/pdf")}
    response = requests.post(url, files=files)
    return response

In [ ]:
file_path = "resumes/Jain, Rainy Resume.pdf"
url = "https://resumeparse-wx2fewkasq-uc.a.run.app/process-resume"

response = opening(file_path, url)

# Parse JSON response
response_data = json.loads(response.text)

# Print the entire response
print("Full Response:")
print(json.dumps(response_data, indent=4))


In [ ]:
import json

def extract_info(text):
    response_data = json.loads(text)
    resume_extracted_info = response_data.get('resume_extracted_info', None)
    return resume_extracted_info
resume_extracted_info = extract_info(response.text)
print(resume_extracted_info)

In [ ]:
def extract_workexp(resume_extracted_info):
    
    # Extract relevant work experience
    work_experience = resume_extracted_info.get('work_exp', [])
    work_exp_job_title = []
    work_exp_description = []
    
    # Print relevant work experience
    for experience in work_experience:
        #print("Job Title:", experience['job_title'])
        work_exp_job_title.append(experience['job_title'])
        #print("Description:", experience['description'])
        work_exp_description.append(experience['description'])
    return work_exp_job_title, work_exp_description

work_exp_job_title, work_exp_description = extract_workexp(resume_extracted_info)
print(work_exp_description)
print()
print(work_exp_job_title)

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("Skills.csv")
df1 = df["Element Name"]
df1

In [ ]:
df1 = df1.unique()

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def find_similarities(df1, work_exp_description):
    
    # Load the model
    model = SentenceTransformer('paraphrase-distilroberta-base-v1')

    # Encode sentences
    max_similarity = -1 
    max_i = -1
    max_j = -1
    for i in range(len(work_exp_description)):
        x = work_exp_description[i]
        for j in range(len(df1)):
            y = df1[j]
            sentences = [x, y]
            
            # Calculate cosine similarity
            sentence_embeddings = model.encode(sentences)
            similarity_score = cosine_similarity([sentence_embeddings[0]], [sentence_embeddings[1]])
            if similarity_score > max_similarity:
                max_similarity = similarity_score
                max_i = i
                max_j = j
            #print(" Similarity Score:", similarity_score[0][0])
            
    return max_similarity, max_i, max_j

max_similarity, max_i, max_j = find_similarities(df1, work_exp_description)
print("Maximum similarity:", max_similarity)
print("Indices (i):", work_exp_description[max_i])
print("Indices (j):",df1[max_j])

In [ ]:
from os import listdir
from os.path import isfile, join
mypath = 'resumes'
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]

In [ ]:
def resume_matching(file_path, url):
    response = opening(file_path, url)
    print(response.text, '\n')
    resume_extracted_info = extract_info(response.text)
    print(resume_extracted_info, '\n')
    work_exp_job_title, work_exp_description = extract_workexp(resume_extracted_info)
    print(work_exp_description, '\n')
    print(work_exp_job_title, '\n')
    max_similarity, max_i, max_j = find_similarities(df1, work_exp_description)
    print("Maximum similarity:", max_similarity)
    print("Indices (i):", work_exp_description[max_i])
    print("Indices (j):",df1[max_j])
    print('-----------------------------------------------------------')

In [ ]:
for i in range(len(onlyfiles)):
    file_path = 'resumes/{}'.format(onlyfiles[i])
    print(file_path)
    resume_matching(file_path, url)

In [ ]:
import requests

url = "https://resumeparse-wx2fewkasq-uc.a.run.app"
resume_endpoint = "/process-resume"
questions_endpoint = "/get-questions"
file_path = "resumes/SiddakMResume2.pdf"

def parse_resume(file_path, url):
    with open(file_path, "rb") as f:
        file_data = f.read()
    files = {"file": (file_path, file_data, "application/pdf")}
    response = requests.post(url + resume_endpoint, files=files)
    return response

def get_generated_questions(url, data):
    # Include required field(s) in the request body
    response = requests.post(url + questions_endpoint, json=data)
    return response.json()

# Parse the resume
response = parse_resume(file_path, url)
if response.status_code == 200:
    print("Resume processed successfully.")
    # Get generated questions
    questions_response = get_generated_questions(url, extract_info(response.text))
    print(questions_response)
    if questions_response.get("message") == "Questions generated successfully.":
        print("Generated Questions:")
        for i, question in enumerate(questions_response["questions"], 1):
            print(f"{i}. {question}")
